In [ ]:
# Phase (I): Test GRPO on Game of 24 — surface drawbacks D1-D4
#
#   D1  CoT redundancy           → track CoT length distribution across training
#   D2  No per-token signal      → (structural; visualised by absence of v_t)
#   D3  Zero-pass@K dead zone    → curate hard puzzles, track which never solve
#   D4  Indiscriminate credit    → inspect failed rollouts that share good prefixes
#
# Phase (II) [later]: swap reward + advantage for velocity / answer-buffer / prefix-buffer
#                     and re-run the same diagnostics to verify the four fixes.

In [ ]:
import os, re, json, random, itertools, math
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
random.seed(0); np.random.seed(0); torch.manual_seed(0)

MODEL_NAME    = "Qwen/Qwen3-0.6B"
OUTPUT_DIR    = Path("output/game24_grpo_baseline")
ROLLOUT_LOG   = OUTPUT_DIR / "rollouts.jsonl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR.resolve())

## 2. Game-of-24 &nbsp;·&nbsp; data &nbsp;·&nbsp; rewards

Everything in this section is a thin wrapper around `src.game24utils`:

- **verifier + solver** — `verify_24`, `enumerate_solutions`
- **puzzle pool + buckets** — `build_puzzle_pool`, `bucket_by_difficulty`
- **splits** — `make_splits` (proportion-based; holds out hard puzzles for D3)
- **prompts / datasets** — `SYSTEM_PROMPT`, `to_chat`, `build_datasets`
- **rewards** — `correctness_reward`, `format_reward`

The notebook itself focuses on training, diagnostics, and the velocity-reward
analysis. Algorithm pieces stay here; bookkeeping pieces live in the module.

In [ ]:
from src.game24utils import (
    verify_24, 
    build_puzzle_pool, bucket_by_difficulty, make_splits,
    to_chat, build_datasets,
    _text,  extract_expr,
    correctness_reward, format_reward,
)

# All solvable 4-tuples from digits 1..9, bucketed by # solutions.
puzzles = build_puzzle_pool(max_n=9)
easy, medium, hard = bucket_by_difficulty(puzzles, easy_min=8, hard_max=2)

print(f"Solvable 4-tuples: {len(puzzles)}")
print(f"  easy   (≥8 sols): {len(easy)}")
print(f"  medium (3-7 sols): {len(medium)}")
print(f"  hard   (≤2 sols): {len(hard)}")
print("example hard puzzle:", hard[0] if hard else None)

In [ ]:
# Proportion-based splits.
# - eval_frac: held-out slice of easy+medium for in-distribution eval
# - probe_frac: slice of HARD puzzles held out as the D3 zero-pass@K probe
train_puzzles, eval_puzzles, hard_probe = make_splits(
    easy, medium, hard, eval_frac=0.10, probe_frac=0.40,
)

print(f"train={len(train_puzzles)}, eval={len(eval_puzzles)}, "
      f"probe(hard, held-out)={len(hard_probe)}")

train_ds, eval_ds, probe_ds = build_datasets(train_puzzles, eval_puzzles, hard_probe)
print(train_ds[0])

In [ ]:
# Rewards live in src.game24utils — trajectory-level only (D2 is structural).
# Sanity:
fake = [[{"role": "assistant", "content": "Let me think...\n#### (3+5)*(7-4)"}]]
print("correctness:", correctness_reward(fake, numbers=[[3, 5, 7, 4]]))
print("format    :", format_reward(fake))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class RolloutLogger:
    """Logs every rollout to JSONL via the reward-fn callback. Returns 0.0 reward."""
    def __init__(self, path: Path, tok):
        self.path = path
        self.tok  = tok
        self.step = 0
        self.path.write_text("")  # truncate

    def __call__(self, completions, numbers, solutions=None, **kwargs):
        with self.path.open("a") as f:
            for i, (c, nums) in enumerate(zip(completions, numbers)):
                text = _text(c)
                expr = extract_expr(text)
                correct = verify_24(list(nums), expr)
                n_tok = len(self.tok.encode(text, add_special_tokens=False))
                f.write(json.dumps({
                    "step": self.step,
                    "idx": i,
                    "numbers": list(nums),
                    "completion": text,
                    "expr": expr,
                    "correct": bool(correct),
                    "n_tokens": int(n_tok),
                }) + "\n")
        self.step += 1
        return [0.0] * len(completions)   # passthrough, contributes nothing

    # Keep TRL happy: it inspects __name__ on reward callables.
    __name__ = "rollout_logger"

rollout_logger = RolloutLogger(ROLLOUT_LOG, tokenizer)
print("Rollout log →", ROLLOUT_LOG)

## 5. Train &nbsp;·&nbsp; vanilla GRPO

Short run (200 steps, 8 generations/prompt) is enough to surface the four
drawbacks. Bump `max_steps` for a longer collapse-watch.

In [ ]:
config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_generations=8,
    max_completion_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=200,
    logging_steps=5,
    bf16=True,
    save_strategy="no",
    report_to="none",
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.4,
)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward, rollout_logger],
    args=config,
    train_dataset=train_ds,
)
trainer.train()
print("Done. Rollouts at:", ROLLOUT_LOG)

## 6. Per-token reward GRPO (implementation pattern)

In [ ]:
import torch
from trl import GRPOTrainer


# -----------------------------------------------------------------------------
# (1) Placeholder per-token reward signal.
#
# Contract:
#   r_t(prompt_ids, completion_ids, completion_mask) -> Tensor of shape (B, T)
#   where T = completion_ids.size(1). Padding positions must be 0
#   (the trainer multiplies by completion_mask anyway, but keep it tidy).
#
# This implementation: r_t = (t / T_eff) * traj_correctness.
# Deliberately silly so we can SEE its effect: if it works, the policy
# should learn to make correct rollouts longer (later tokens earn more).
# -----------------------------------------------------------------------------
def placeholder_per_token_reward(
    prompt_ids: torch.Tensor,         # (B, P)
    completion_ids: torch.Tensor,     # (B, T)
    completion_mask: torch.Tensor,    # (B, T) -- 1 for real tokens, 0 for pad
    *,
    tokenizer,
    numbers_per_row,                  # list[list[int]] aligned with B
) -> torch.Tensor:
    B, T = completion_ids.shape
    device = completion_ids.device

    # 1. trajectory-level correctness (one scalar per row).
    traj_r = torch.zeros(B, device=device)
    for i in range(B):
        text = tokenizer.decode(
            completion_ids[i][completion_mask[i].bool()],
            skip_special_tokens=True,
        )
        expr = extract_expr(text)
        traj_r[i] = float(verify_24(list(numbers_per_row[i]), expr))

    # 2. position ramp: r_t = (t / T_eff) -- later tokens worth more.
    T_eff = completion_mask.sum(dim=1).clamp(min=1).float()    # (B,)
    pos = torch.arange(T, device=device, dtype=torch.float32)  # (T,)
    ramp = pos.unsqueeze(0) / T_eff.unsqueeze(1)               # (B, T)

    # 3. combine; mask padding.
    r_t = ramp * traj_r.unsqueeze(1)
    return r_t * completion_mask.float()


# -----------------------------------------------------------------------------
# (2) Trainer subclass: per-token advantage = z-score over ALL valid completion
#     tokens in the microbatch. "This token is informative compared to the
#     other tokens we saw in this batch."
# -----------------------------------------------------------------------------
class PerTokenAdvantageTrainer(GRPOTrainer):
    def __init__(self, *args, per_token_reward_fn=None, reward_kwargs_fn=None, **kw):
        super().__init__(*args, **kw)
        self.per_token_reward_fn = per_token_reward_fn
        self.reward_kwargs_fn = reward_kwargs_fn or (lambda inputs: {})

    def _compute_loss(self, model, inputs):
        if self.per_token_reward_fn is None:
            return super()._compute_loss(model, inputs)

        prompt_ids     = inputs["prompt_ids"]
        completion_ids = inputs["completion_ids"]   # (Bp, T), right-padded with pad_id
        mask           = inputs["completion_mask"]  # (Bp, T), 1 for real tokens

        with torch.no_grad():
            r_t = self.per_token_reward_fn(
                prompt_ids, completion_ids, mask,
                tokenizer=self.processing_class,
                **self.reward_kwargs_fn(inputs),
            )  # (Bp, T)
            assert r_t.shape == completion_ids.shape, \
                f"per-token reward must be (Bp, T)={completion_ids.shape}, got {r_t.shape}"

            m         = mask.bool()
            pool      = r_t[m]
            mu, sd    = pool.mean(), pool.std()
            adv       = ((r_t - mu) / (sd + 1e-6)) * m

        inputs = dict(inputs)
        inputs["advantages"] = adv
        return super()._compute_loss(model, inputs)


# -----------------------------------------------------------------------------
# (3) Wire it up exactly like cell 8, but with the per-token trainer.
# -----------------------------------------------------------------------------
PT_OUTPUT_DIR = Path("output/game24_grpo_pertoken")
PT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pt_config = GRPOConfig(
    output_dir=str(PT_OUTPUT_DIR),
    num_generations=8,
    max_completion_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=200,
    logging_steps=5,
    bf16=True,
    save_strategy="no",
    report_to="none",
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.4,
)

pt_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)

# `numbers` rides along on each dataset row; the trainer puts it under
# inputs["numbers"] so the reward fn can recover the 4-tuple per rollout.
pt_trainer = PerTokenAdvantageTrainer(
    model=pt_model,
    reward_funcs=[correctness_reward, format_reward],   # still trajectory-level loggers
    per_token_reward_fn=placeholder_per_token_reward,
    reward_kwargs_fn=lambda inputs: {"numbers_per_row": inputs["numbers"]},
    args=pt_config,
    train_dataset=train_ds,
)
pt_trainer.train()
print("Done. Output:", PT_OUTPUT_DIR)


/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_ref" in ModelCreateRes has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_output" in EvalResultsTrial has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/pyd

NameError: name 'Path' is not defined